In [1]:
from google.cloud import bigquery
from pathlib import Path
import os
import requests
import pandas as pd
import db_dtypes
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support, brier_score_loss
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
import model_config 
import importlib

In [8]:
def load_data():

    path = Path("D:\Python Projects\Hospital readmission risk\index_stay.csv")

    if path.exists():
        return pd.read_csv(path, sep = ',')


    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"D:\Python Projects\Hospital readmission risk\.secrets\hospital-readmission-4-code.json"
    client = bigquery.Client(project = "hospital-readmission-4")
    sql = """
    SELECT
        *
        from `hospital-readmission-4.helper_tables.index_stay`
    """

    job = client.query(sql)
    rows = list(job.result())


    data_raw = [dict(r) for r in rows]
    return pd.DataFrame(data_raw)

In [9]:
def build_preprocessor(df_raw):

    analysis_cols = [
    'patient_age', 'gender', 'length_of_stay', 'main_code', 'num_diagnoses',
    'num_chronic_conditions', 'num_procedures', 'has_diabetes', 'has_cancer',
    'has_hiv', 'has_hf', 'has_alz', 'has_ckd', 'had_surgery', 'admission_cost',
    'total_procedure_costs', 'total_medication_costs', 'total_stay_cost', 
    'admissions_365d', 'tot_length_of_stay_365d', 'avg_cost_of_prev_stays',
    'is_planned', 'following_unplanned_admission_flag', 'readmit_30d', 'readmit_90d'
    ]

    df_numeric = df_raw[analysis_cols]

    df_numeric['avg_cost_of_prev_stays'] = df_numeric['avg_cost_of_prev_stays'].fillna(0)
    df_numeric['following_unplanned_admission_flag'] = df_numeric['following_unplanned_admission_flag'].fillna(0)

    df_numeric = pd.get_dummies(df_numeric)
    df_numeric = df_numeric.drop(columns = 'gender_F')

    mask = df_numeric['following_unplanned_admission_flag'] == 0
    df_numeric.loc[mask, ['readmit_30d', 'readmit_90d']] = 0
    mask = df_numeric['readmit_90d'] == 0
    df_numeric.loc[mask, 'following_unplanned_admission_flag'] = 0

    df_results = df_numeric[['readmit_30d', 'readmit_90d']]
    df_numeric['log_stay_cost'] = np.log(df_numeric['total_stay_cost'])
    df_numeric['log_prev_avg_stay_cost'] = np.log1p(df_numeric['avg_cost_of_prev_stays'])
    df_numeric['log_total_procedure_costs'] = np.log1p(df_numeric['total_procedure_costs'])
    df_numeric['log_total_medication_costs'] = np.log1p(df_numeric['total_medication_costs'])

    df_numeric = df_numeric.drop(columns = ['total_stay_cost', 'avg_cost_of_prev_stays', 'total_procedure_costs', 'total_medication_costs'
,'readmit_30d', 'readmit_90d', 'following_unplanned_admission_flag', 'main_code'])

    return df_numeric, df_results

In [10]:
def make_train_test_split(X, y):

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

    return X_train, X_test, y_train, y_test

In [11]:
def train_model(X, y, model_name, model, params, test_log, d30 = True):

    model_name = model_name + ('_d30' if d30 else '_d90')

    model = model.set_params(**params)

    pipe = Pipeline([('scaler', StandardScaler()), (model_name, model)])
    """
    scoring = ["roc_auc", "average_precision"]

    cv_results = cross_validate(
        estimator=pipe,
        X=X,
        y=y,
        cv=5,
        scoring=scoring,
        return_train_score=False,
    )

    test_log.loc[model_name, 'roc_auc'] = cv_results["test_roc_auc"].mean()
    test_log.loc[model_name, 'roc_auc_std'] = cv_results["test_roc_auc"].std()
    test_log.loc[model_name, 'pr_auc'] = cv_results["test_average_precision"].mean()
    test_log.loc[model_name, 'pr_auc_std'] = cv_results["test_average_precision"].std()
    """
    pipe.fit(X,y)
    
    return pipe, model_name, test_log
    

In [12]:
def evaluate_model(X, y_true, model_name, pipe, coefs, metrics):


    y_proba = pipe.predict_proba(X)[:,1]
    y_pred =  pipe.predict(X)

    metrics.loc[model_name, 'roc'] = roc_auc_score(y_true, y_proba)
    metrics.loc[model_name, 'pr'] = average_precision_score(y_true, y_proba)
    metrics.loc[model_name, 'brier_loss_total'] = brier_score_loss(y_true, y_proba)
    metrics.loc[model_name, 'brier_loss_half'] = brier_score_loss(y_true, np.array(y_proba) > 0.5)

    precision, recall, f1, _= precision_recall_fscore_support(y_true, y_pred, average="binary")
    metrics.loc[model_name, 'precision'] = precision
    metrics.loc[model_name, 'recall'] = recall
    metrics.loc[model_name, 'f1'] = f1

    est = pipe.named_steps[model_name]

    if isinstance(est, LogisticRegression):
    # est.coef_.shape == (1, n_features) for binary
        coefs[model_name] = est.coef_[0]

    elif hasattr(est, "feature_importances_"):
    # trees, random forest, gradient boosting
        coefs[model_name] = est.feature_importances_

    return coefs, metrics



rerun and remove main_code from numeric_columns

In [13]:
df_raw = load_data()
df_numeric, df_results = build_preprocessor(df_raw)

C:\Users\4infi\AppData\Local\Temp\ipykernel_9516\2591555852.py:6: DtypeWarning: Columns (37) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, sep = ',')
C:\Users\4infi\AppData\Local\Temp\ipykernel_9516\2235083803.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_numeric['avg_cost_of_prev_stays'] = df_numeric['avg_cost_of_prev_stays'].fillna(0)
C:\Users\4infi\AppData\Local\Temp\ipykernel_9516\2235083803.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-co

In [14]:
models = {
    'logreg': LogisticRegression(),
    'rf': RandomForestClassifier(),
    'lightgbm': LGBMClassifier()
}

test_log = pd.DataFrame(columns = ['roc_auc', 'pr_auc'])

metrics_log = pd.DataFrame(columns = ['roc', 'pr', 'brier_loss_total', 'brier_loss_half', 'precision', 'recall', 'f1'])

params = {'class_weight': 'balanced', 'solver': 'saga', 'max_iter': 2000}

coefs = pd.DataFrame(index = df_numeric.columns)

for model_name in models:

    for set in model_config.MODEL_CONFIGS:

        if set['name'] == model_name:

            params = set['params']

    X_train, X_test, y_train, y_test = make_train_test_split(df_numeric, df_results['readmit_30d'])
    
    trained_pipe, name, test_log = train_model(X_train, y_train, model_name, models[model_name], params, test_log)

    coefs, metrics_log = evaluate_model(X_test, y_test, name, trained_pipe, coefs, metrics_log)

    X_train, X_test, y_train, y_test = make_train_test_split(df_numeric, df_results['readmit_90d'])
    
    trained_pipe, name, test_log = train_model(X_train, y_train, model_name, models[model_name], params, test_log, d30 = False)

    coefs, metrics_log = evaluate_model(X_test, y_test, name, trained_pipe, coefs, metrics_log)






[LightGBM] [Info] Number of positive: 4119, number of negative: 83483
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002805 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1430
[LightGBM] [Info] Number of data points in the train set: 87602, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 8044, number of negative: 79558
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002647 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1430
[LightGBM] [Info] Number of data points in the train set: 87602, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [22]:
metrics_log


,roc,pr,brier_loss_total,brier_loss_half,precision,recall,f1
logreg_d30,0.919753,0.440797,0.091,0.091503,0.318764,0.8119,0.457792
logreg_d90,0.91297,0.741169,0.079562,0.076937,0.558737,0.759102,0.643688
rf_d30,0.940802,0.634636,0.027306,0.038537,0.614319,0.510557,0.557652
rf_d90,0.951598,0.831835,0.028867,0.033423,0.866436,0.750623,0.804383
lightgbm_d30,0.949856,0.645473,0.058289,0.074015,0.37644,0.846449,0.521123
lightgbm_d90,0.955166,0.837012,0.048901,0.058673,0.640515,0.818454,0.718634
